Cleaning Data

In [2]:
import os
import librosa
import soundfile as sf
import numpy as np
from tqdm import tqdm

# تنظیمات اصلی
INPUT_DIR = './dataset'  # فرض بر این است که پوشه‌های German, Spanish و... اینجا هستند
OUTPUT_DIR = './processed_dataset'
TARGET_SR = 22050  # نرخ نمونه‌برداری استاندارد
CHUNK_LENGTH = None  # طول هر قطعه به ثانیه (اگر نمی‌خواهید قطعه‌بندی کنید، این را None بگذارید)
OVERLAP = 1.0       # همپوشانی قطعات (اختیاری)

def ensure_dir(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)

def clean_and_process_audio(file_path, save_dir, filename):
    try:
        # 1. بارگذاری و تبدیل به مونو و تغییر نرخ نمونه‌برداری
        # librosa به صورت پیش‌فرض mono=True است
        y, sr = librosa.load(file_path, sr=TARGET_SR, mono=True)

        # 2. حذف سکوت (Silence Removal)
        # top_db=20 یعنی صداهای زیر 20 دسی‌بل نسبت به پیک را سکوت در نظر بگیر
        y_trimmed, _ = librosa.effects.trim(y, top_db=20)
        
        # اگر فایل بعد از حذف سکوت خیلی کوتاه شد، نادیده بگیر
        if len(y_trimmed) < TARGET_SR: 
            return

        # 3. نرمال‌سازی (Normalization)
        # نرمال‌سازی به بازه -1 تا 1
        max_val = np.max(np.abs(y_trimmed))
        if max_val > 0:
            y_norm = y_trimmed / max_val
        else:
            y_norm = y_trimmed

        # 4. قطعه‌بندی (Segmentation) - اختیاری اما توصیه شده برای افزایش دیتا
        if CHUNK_LENGTH:
            chunk_samples = int(CHUNK_LENGTH * TARGET_SR)
            step_samples = int((CHUNK_LENGTH - OVERLAP) * TARGET_SR)
            
            num_chunks = 0
            # تقسیم فایل طولانی به قطعات کوچک
            for i in range(0, len(y_norm) - chunk_samples + 1, step_samples):
                chunk = y_norm[i : i + chunk_samples]
                
                # ذخیره قطعه
                chunk_name = f"{os.path.splitext(filename)[0]}_part{num_chunks}.wav"
                sf.write(os.path.join(save_dir, chunk_name), chunk, TARGET_SR)
                num_chunks += 1
        else:
            # اگر قطعه‌بندی نخواهیم، کل فایل تمیز شده را ذخیره می‌کنیم
            sf.write(os.path.join(save_dir, filename), y_norm, TARGET_SR)

    except Exception as e:
        print(f"Error processing {file_path}: {e}")

def main():
    languages = ['German', 'Korean', 'Spanish', 'Italian']
    
    print("شروع عملیات Data Cleaning...")
    
    for lang in languages:
        input_lang_dir = os.path.join(INPUT_DIR, lang)
        output_lang_dir = os.path.join(OUTPUT_DIR, lang)
        
        if not os.path.exists(input_lang_dir):
            print(f"هشدار: پوشه {input_lang_dir} پیدا نشد.")
            continue
            
        ensure_dir(output_lang_dir)
        
        files = [f for f in os.listdir(input_lang_dir) if f.endswith(('.mp3', '.wav', '.flac'))]
        print(f"در حال پردازش {lang} ({len(files)} فایل)...")
        
        for f in tqdm(files):
            clean_and_process_audio(
                os.path.join(input_lang_dir, f),
                output_lang_dir,
                f
            )

    print("\nپاک‌سازی تمام شد. فایل‌های پردازش شده در پوشه processed_dataset ذخیره شدند.")

if __name__ == "__main__":
    main()

شروع عملیات Data Cleaning...
در حال پردازش German (172 فایل)...


  0%|          | 0/172 [00:00<?, ?it/s]c:\Users\SAH82\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 172/172 [01:11<00:00,  2.40it/s]


در حال پردازش Korean (180 فایل)...


100%|██████████| 180/180 [00:48<00:00,  3.74it/s]


در حال پردازش Spanish (180 فایل)...


100%|██████████| 180/180 [00:49<00:00,  3.65it/s]


در حال پردازش Italian (180 فایل)...


100%|██████████| 180/180 [00:53<00:00,  3.36it/s]


پاک‌سازی تمام شد. فایل‌های پردازش شده در پوشه processed_dataset ذخیره شدند.


Feature

In [19]:
import os
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm

# تنظیمات مسیرها
INPUT_DIR = './processed_dataset'
OUTPUT_FILE = 'features.csv'

# لیست دقیق زبان‌ها (مطابق با نام پوشه‌های ساخته شده در مرحله قبل)
TARGET_LANGUAGES = ['German', 'Italian', 'Spanish', 'Korean']

def extract_features(file_path):
    try:
        # بارگذاری فایل صوتی (با نرخ نمونه‌برداری اصلی فایل که قبلا تنظیم شده)
        y, sr = librosa.load(file_path, sr=None)

        features = {}

        # 1. MFCC (میانگین و انحراف معیار)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        for i in range(mfcc.shape[0]):
            features[f'mfcc_mean_{i+1}'] = np.mean(mfcc[i])
            features[f'mfcc_std_{i+1}'] = np.std(mfcc[i])

        # 2. Spectral Centroid
        spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)
        features['spectral_centroid_mean'] = np.mean(spec_cent)
        features['spectral_centroid_std'] = np.std(spec_cent)

        # 3. Spectral Bandwidth
        spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)
        features['spectral_bandwidth_mean'] = np.mean(spec_bw)
        features['spectral_bandwidth_std'] = np.std(spec_bw)

        # 4. Spectral Rolloff
        spec_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        features['spectral_rolloff_mean'] = np.mean(spec_rolloff)
        features['spectral_rolloff_std'] = np.std(spec_rolloff)

        # 5. Zero Crossing Rate
        zcr = librosa.feature.zero_crossing_rate(y)
        features['zcr_mean'] = np.mean(zcr)
        features['zcr_std'] = np.std(zcr)
        
        # 6. RMS Energy
        rms = librosa.feature.rms(y=y)
        features['rms_mean'] = np.mean(rms)
        features['rms_std'] = np.std(rms)

        return features

    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

def main():
    all_data = []
    
    print(f"شروع استخراج ویژگی‌ها از پوشه: {INPUT_DIR}")
    
    # حلقه اصلی: پیمایش روی لیست زبان‌های مشخص شده
    for lang in TARGET_LANGUAGES:
        # ساخت مسیر پوشه زبان خاص
        lang_dir = os.path.join(INPUT_DIR, lang)
        
        # بررسی اینکه آیا پوشه واقعا وجود دارد یا نه
        if not os.path.exists(lang_dir):
            print(f"هشدار: پوشه {lang} در مسیر {lang_dir} پیدا نشد!")
            continue
            
        # لیست کردن تمام فایل‌های wav داخل آن پوشه
        files = [f for f in os.listdir(lang_dir) if f.endswith('.mp3')] ########### wav -> mp3
        print(f"در حال پردازش زبان: {lang} - تعداد فایل: {len(files)}")
        
        # پردازش تک‌تک فایل‌های این زبان
        for f in tqdm(files, desc=f"Extracting {lang}"):
            file_path = os.path.join(lang_dir, f)
            
            # فراخوانی تابع استخراج ویژگی
            features = extract_features(file_path)
            
            if features:
                # اضافه کردن برچسب (Label) که همان نام زبان است
                features['label'] = lang
                # ذخیره نام فایل محض اطمینان (اختیاری)
                features['filename'] = f
                all_data.append(features)

    # ذخیره نهایی در CSV
    if all_data:
        df = pd.DataFrame(all_data)
        
        # مرتب‌سازی ستون‌ها (label را به آخر می‌بریم)
        cols = [c for c in df.columns if c not in ['label', 'filename']]
        df = df[cols + ['label']]
        
        df.to_csv(OUTPUT_FILE, index=False)
        print(f"\nپایان! ویژگی‌ها در فایل '{OUTPUT_FILE}' ذخیره شدند.")
        print(f"ابعاد دیتاست نهایی: {df.shape}")
    else:
        print("هیچ داده‌ای استخراج نشد. لطفا مسیرها را چک کنید.")

if __name__ == "__main__":
    main()

شروع استخراج ویژگی‌ها از پوشه: ./processed_dataset
در حال پردازش زبان: German - تعداد فایل: 172


Extracting German: 100%|██████████| 172/172 [01:00<00:00,  2.82it/s]


در حال پردازش زبان: Italian - تعداد فایل: 180


Extracting Italian: 100%|██████████| 180/180 [01:09<00:00,  2.58it/s]


در حال پردازش زبان: Spanish - تعداد فایل: 180


Extracting Spanish: 100%|██████████| 180/180 [01:01<00:00,  2.91it/s]


در حال پردازش زبان: Korean - تعداد فایل: 180


Extracting Korean: 100%|██████████| 180/180 [01:01<00:00,  2.92it/s]


پایان! ویژگی‌ها در فایل 'features.csv' ذخیره شدند.
ابعاد دیتاست نهایی: (712, 37)
